# 02 - LSST OpSim footprint & observation counts

Explores the LSST OpSim cadence: survey timeline, which random sky pointings
fall inside the footprint, and how the number of observations (per pointing)
separates the Wide-Fast-Deep, Deep Drilling Fields and galactic plane.

Uses `cmsne.opsim`, `cmsne.observations` and `cmsne.config`.

**Note:** set `cmsne.config.MY_OPSIM_DB` to your OpSim `.db` file first.

In [ ]:
# If running in Colab, install the dependencies (uncomment):
# !pip install sncosmo
# !pip install git+https://github.com/LSSTDESC/OpSimSummaryV2.git

# Make the `cmsne` package importable when this notebook lives in notebooks/.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
from astropy.time import Time

from cmsne.config import survey_dates
from cmsne.opsim import (load_opsim_survey, create_sky_pointings,
                         initialise_opsim_summary, get_Nobs_MJD)
from cmsne.observations import (GeneratorWrapper, opsim_observation,
                                select_observation_time_period)

## Survey timeline

In [ ]:
survey = load_opsim_survey()

earliest_mjd = survey.opsimdf['observationStartMJD'].min()
latest_mjd = survey.opsimdf['observationStartMJD'].max()

print(f"Earliest MJD in entire survey: {earliest_mjd}")
print(f"Latest MJD in entire survey: {latest_mjd}")
print(f"Survey duration: {latest_mjd - earliest_mjd:.1f} days ({(latest_mjd - earliest_mjd)/365.25:.1f} years)")
print()

start_time = Time(earliest_mjd, format='mjd')
start_date = start_time.datetime

print("LSST survey dates from OpSim data:")
print()
for year in range(10):
    year_mjd = earliest_mjd + (year * 365.25)
    year_time = Time(year_mjd, format='mjd')
    year_date = year_time.datetime
    print(f"Year {year:<2} - {year_date.strftime('%d/%m/%Y')} - MJD {year_mjd:.0f}")

print()
print("Actual survey end:")
end_time = Time(latest_mjd, format='mjd')
end_date = end_time.datetime
print(f"End    - {end_date.strftime('%d/%m/%Y')} - MJD {latest_mjd:.0f}")

`survey_dates` (start MJD of each observing year) is defined in `cmsne.config`:

In [ ]:
print(survey_dates)

## Which random pointings fall inside the footprint?

In [ ]:
ra, dec = create_sky_pointings(10000)

# Initialise opsim generator
gen = initialise_opsim_summary(ra, dec)

# Plot lsst pointings
opsim_ra_list, opsim_dec_list = [], []
ra_excluded, dec_excluded = [], []
opsim_Nobs = []

for p in range(len(ra)):

    obs = next(gen)

    opsim_ra = np.mean(obs['fieldRA'])
    opsim_dec = np.mean(obs['fieldDec'])

    Nobs = obs['expMJD']

    if np.isnan(opsim_ra) or np.isnan(opsim_dec):
        ra_excluded.append(ra[p])
        dec_excluded.append(dec[p])
        continue

    opsim_ra_list.append(opsim_ra)
    opsim_dec_list.append(opsim_dec)
    opsim_Nobs.append(len(Nobs))

opsim_ra_list = np.array(opsim_ra_list)
opsim_dec_list = np.array(opsim_dec_list)
opsim_Nobs = np.array(opsim_Nobs)

opsim_ra1 = coord.Angle(opsim_ra_list * u.degree)
opsim_dec = coord.Angle(opsim_dec_list * u.degree)
opsim_ra = - opsim_ra1.wrap_at(180*u.degree)

ra_excluded = np.array(ra_excluded)
dec_excluded = np.array(dec_excluded)

ra_excluded1 = coord.Angle(ra_excluded * u.degree)
dec_excluded = coord.Angle(dec_excluded * u.degree)
ra_excluded = - ra_excluded1.wrap_at(180*u.degree)

fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(221, projection="mollweide")
im1 = ax1.scatter(ra_excluded.radian, dec_excluded.radian, s=0.5, label='excluded')
im2 = ax1.scatter(opsim_ra.radian, opsim_dec.radian, s=0.5, label='included')
plt.legend(loc='upper right', fontsize=12)

## Single-pointing observation history

In [ ]:
gen = initialise_opsim_summary(90, -30)

# Wrap the raw generator so we can re-read its current pointing.
gen_wrapper_for_cell_test = GeneratorWrapper(gen)

observations = opsim_observation(gen_wrapper_for_cell_test, use_previous=True)
if observations is not None:
    print(observations)

    plt.figure(figsize=(8, 3))
    plt.hist(observations.opsim_times, bins=100)
    plt.title("Histogram of visits for 1 event", fontsize=18)
    plt.xlabel("MJD")
    plt.show()
else:
    print("No observations found for the given pointing.")

In [ ]:
observations_3yr = select_observation_time_period(observations, 60220)
plt.figure(figsize=(8, 3))
plt.hist(observations_3yr.opsim_times, bins=100)
plt.title("Histogram of observations for 1 event up to year 3", fontsize=18)
plt.xlabel("MJD")
plt.show()

## Number of observations across the sky

In [ ]:
ra, dec = create_sky_pointings(100000)
gen = initialise_opsim_summary(ra, dec)
result_3 = get_Nobs_MJD(ra, dec, gen, MJD=survey_dates[3])
ra_3, dec_3, Nobs_3, Nobs_3_10 = result_3['opsim_ra'], result_3['opsim_dec'], result_3['Nobs'], result_3['Nobs_10']

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
plt.hist(Nobs_3_10, bins=100, alpha=0.6, range=(0, 1250))
plt.ylabel('Counts', fontsize=20, labelpad=20)
plt.xlabel(r'$N_{\rm obs}$ after 10 years', fontsize=20, labelpad=10)
plt.axvline(x=400, ls='--', color='C3', lw=2)
plt.text(0.4, 0.8, 'WFD', fontsize=30, color='C3', transform=ax.transAxes)
plt.text(0.05, 0.8, 'galactic plane', fontsize=20, color='C3', transform=ax.transAxes)
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
plt.hist(Nobs_3_10, bins=70, alpha=0.6, range=(0, 14000))
plt.ylabel('Counts', fontsize=20, labelpad=20)
plt.xlabel(r'$N_{\rm obs}$ after 10 years', fontsize=20, labelpad=10)
plt.axvline(x=1000, ls='--', color='C3', lw=2)
plt.text(0.5, 0.6, 'DDF', fontsize=40, color='C3', transform=ax.transAxes)
plt.yscale('log')
plt.show()

In [ ]:
DDF_ra = coord.Angle(np.array([345.97, 31.04, 40.29, 150.70, 61.24]), u.degree)
DDF_dec = coord.Angle(np.array([-43.18, -17.90, -45.47, -9.39, -48.42]), u.degree)
DDF_ra = coord.Angle(np.array([150.08, 53.17, 59.41, 63.11, 9.43, 35.74]), u.degree)
DDF_dec = coord.Angle(np.array([2.16, -28.03, -49.15, -47.79, -43.94, -4.75]), u.degree)
DDF_ra = - DDF_ra.wrap_at(180*u.degree)

In [ ]:
fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(221, projection="mollweide")
im1 = ax1.scatter(ra_3.radian, dec_3.radian, s=1, c='limegreen', label='WFD')
im1 = ax1.scatter(ra_3.radian[Nobs_3_10 > 1040], dec_3.radian[Nobs_3_10 > 1040], s=15, c='red', label='DDF')
im1 = ax1.scatter(ra_3.radian[Nobs_3_10 < 400], dec_3.radian[Nobs_3_10 < 400], s=1, c='midnightblue', label='galactic plane')
ax1.plot(DDF_ra.radian, DDF_dec.radian, '*', color='yellow', ms=5, label='DDF coords')
plt.legend(loc=(0.95, 0.8), fontsize=12, facecolor='grey')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
plt.hist(Nobs_3, bins=100, alpha=0.6, range=(0, 400), label="all")
plt.hist(Nobs_3[Nobs_3_10 > 400], bins=100, alpha=0.8, range=(0, 400), color='C1', label="WFD")
plt.ylabel('Counts', fontsize=20, labelpad=20)
plt.xlabel(r'$N_{\rm obs}$ after 3 years', fontsize=20, labelpad=10)
plt.legend(fontsize=18)
plt.axvline(x=300, ls='--', color='C3', lw=2)
plt.text(0.4, 0.9, 'background', fontsize=20, color='C3', transform=ax.transAxes)
plt.text(0.8, 0.5, 'active', fontsize=20, color='C3', transform=ax.transAxes)

In [ ]:
fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(221, projection="mollweide")
im1 = ax1.scatter(ra_3.radian[Nobs_3_10 > 400], dec_3.radian[Nobs_3_10 > 400], s=5, c='C0', label='Background')
im1 = ax1.scatter(ra_3.radian[Nobs_3 > 300], dec_3.radian[Nobs_3 > 300], s=5, c='C1', label='Active')
plt.legend(loc=(0.95, 0.8), fontsize=12)

## Observation counts per year

In [ ]:
MJD_years = survey_dates[1:]
gen = initialise_opsim_summary(ra, dec)

# Use optimized approach with list of MJDs
results_list = get_Nobs_MJD(ra, dec, gen, MJD=MJD_years)

# Extract data for plotting
ra_years = [result['opsim_ra'] for result in results_list]
dec_years = [result['opsim_dec'] for result in results_list]
Nobs_years = [result['Nobs'] for result in results_list]
Nobs_10_years = [result['Nobs_10'] for result in results_list]

# Create masks to separate different regions
fig = plt.figure(figsize=(12, 20))
for i, (Nobs, Nobs_10) in enumerate(zip(Nobs_years, Nobs_10_years)):
    ax1 = fig.add_subplot(5,2,i+1, projection="mollweide")

    ddf_mask = Nobs_10 > 1000
    galactic_mask = Nobs_10 < 400
    wfd_mask = ~ddf_mask & ~galactic_mask  # WFD: 400 <= Nobs_10 <= 1000

    if np.any(wfd_mask):
        im1 = ax1.scatter((ra_years[i][wfd_mask]).radian, (dec_years[i][wfd_mask]).radian,
                         s=1, c=Nobs[wfd_mask], cmap='viridis')
        fig.colorbar(im1, ax=ax1, label="num observations", fraction=0.025)

    if np.any(galactic_mask):
        ax1.scatter((ra_years[i][galactic_mask]).radian, (dec_years[i][galactic_mask]).radian,
                   s=1, c='grey', alpha=0.6, label='Galactic plane')

    if np.any(ddf_mask):
        ax1.scatter((ra_years[i][ddf_mask]).radian, (dec_years[i][ddf_mask]).radian,
                   s=3, c='red', label='DDF')

    if np.any(ddf_mask) or np.any(galactic_mask):
        ax1.legend(loc='upper right', fontsize=8)

    ax1.set_title("Year " + str(i+1), fontsize=18)